In [3]:
from typing import TypedDict
from langgraph.graph import START, StateGraph, END
import os
import requests
from dotenv import load_dotenv

load_dotenv()

SAIS_APP_URL = os.getenv("SAIS_APP_URL")
SAIS_TOKEN = os.getenv("SAIS_TOKEN")
SAIS_MODEL_GENERATOR = os.getenv("SAIS_MODEL_GENERATOR", "gpt-4")
SAIS_MODEL_EVALUATOR = os.getenv("SAIS_MODEL_EVALUATOR", "gpt-4.1")

if not SAIS_APP_URL or not SAIS_TOKEN:
    raise SystemExit("Error: SAIS_APP_URL and SAIS_TOKEN environment variables must be set.")

PROXY_HOST = os.getenv("PROXY_HOST")
PROXY_PORT = os.getenv("PROXY_PORT")
PROXY_ENABLED = os.getenv("PROXY_ENABLED", "false").lower() == "true"
SSL_VERIFY = os.getenv("SSL_VERIFY", "true").lower() == "true"

def generate_answer(context: str, query: str, model: str = None) -> str:
    selected_model = model or SAIS_MODEL_GENERATOR

    response = requests.post(
        f"{SAIS_APP_URL}/v1/responses",
        headers={
            "Authorization": f"Bearer {SAIS_TOKEN}",
            "Content-Type": "application/json",
            "ApplicationType": "BRProduct",
        },
        json={
            "model": selected_model,
            "instructions": "You are an AI technical support assistant.\n\nAnswer the user's question to the best of your knowledge.",
            "input": context + "\n\n" + query,
        },
        verify=SSL_VERIFY,
        proxies={
            "http": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
            "https": f"http://{PROXY_HOST}:{PROXY_PORT}" if PROXY_ENABLED else None,
        }
    )

    if response.status_code == 200:
        response_data = response.json()
        try:
            outputs = response_data["body"]["output"]
            texts = []
            for item in outputs:
                if item.get("type") == "message":
                    for content in item.get("content", []):
                        if content.get("type") == "output_text":
                            texts.append(content.get("text", ""))
            if texts:
                return "\n".join(texts)
            return "Error: No output text found in the response."
        except KeyError:
            return "Error: Unexpected response format."
    else:
        return f"Error: Request failed with status code {response.status_code}"

class Mystate(TypedDict):
    topic: str
    tweet: str
    noofiterations: int
    maximumiterations: int

def generate_tweet(state: Mystate) -> dict:
    tweet = generate_answer(
        state["topic"],
        f"Generate a tweet about {state['topic']}.",
        model=SAIS_MODEL_GENERATOR
    )
    return {"tweet": tweet, "noofiterations": state["noofiterations"] + 1}

def evaluate_tweet(state: Mystate) -> dict:
    PROMPT = (
        "You are an expert Social Media Critic and Copywriter. Your task is to evaluate a generated tweet based on a specific topic and determine if it is ready for publication."
        " Analyze the provided tweet and assign a final status of either 'APPROVED' or 'REJECTED'."
        " Evaluation Criteria:"
        " Relevance: Does the tweet align directly with the core topic?"
        " Hook & Engagement: Is the first line punchy enough to stop someone scrolling?"
        " Clarity & Brevity: Is the message clear, concise, and comfortably under the 280-character limit?"
        " Tone: Is it conversational, authoritative, yet engaging (no corporate fluff or excessive jargon)?"
        " Formatting: Are emojis used tastefully (maximum 2)? Are hashtags minimal and highly relevant (maximum 2)?"
    )

    evaluation_prompt = (
        f"{PROMPT}\n\nTopic: {state['topic']}\nTweet: {state['tweet']}\n\n"
        "Please provide your evaluation and final status."
    )

    evaluation_result = generate_answer(
        state["topic"],
        evaluation_prompt,
        model=SAIS_MODEL_EVALUATOR
    )

    if "APPROVED" in evaluation_result.upper():
        return {"tweet": state["tweet"] + "\n\nStatus: APPROVED"}
    else:
        return {"tweet": state["tweet"] + "\n\nStatus: REJECTED"}

def route_after_evaluation(state: Mystate) -> str:
    if "APPROVED" in state["tweet"].upper():
        return "end"
    if state["noofiterations"] >= state["maximumiterations"]:
        return "end"
    return "optimize"

def optimize_tweet(state: Mystate) -> dict:
    optimization_prompt = (
        "You are an expert Social Media Critic and Copywriter. Your task is to optimize a generated tweet based on a specific topic and the provided evaluation feedback."
        " Analyze the provided tweet and make necessary improvements to enhance its relevance, engagement, clarity, tone, and formatting."
        " Ensure the optimized tweet is under 280 characters, maintains a conversational tone, and includes tasteful use of emojis (maximum 2) and relevant hashtags (maximum 2)."
    )
    optimization_prompt += f"\n\nTopic: {state['topic']}\nTweet: {state['tweet']}\n\nPlease provide your optimized version of the tweet."
    optimized_tweet = generate_answer(
        state["topic"],
        optimization_prompt,
        model=SAIS_MODEL_GENERATOR
    )
    return {"tweet": optimized_tweet, "noofiterations": state["noofiterations"] + 1}

graph = StateGraph(Mystate)

graph.add_node("generate_tweet", generate_tweet)
graph.add_node("evaluate_tweet", evaluate_tweet)
graph.add_node("optimize_tweet", optimize_tweet)

graph.add_edge(START, "generate_tweet")
graph.add_edge("generate_tweet", "evaluate_tweet")
graph.add_conditional_edges("evaluate_tweet", route_after_evaluation, {
    "optimize": "optimize_tweet",
    "end": END
})
graph.add_edge("optimize_tweet", "evaluate_tweet")

app = graph.compile()

initial_state = {
    "topic": "Artificial Intelligence",
    "tweet": "",
    "noofiterations": 0,
    "maximumiterations": 3
}

print("Initial State:", initial_state)
result = app.invoke(initial_state)
print("Final State:", result)
# ...existing code.

Initial State: {'topic': 'Artificial Intelligence', 'tweet': '', 'noofiterations': 0, 'maximumiterations': 3}


c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\rajpri\Documents\GenerativeAI\.venv\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host '10.98.21.23'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\r

Final State: {'topic': 'Artificial Intelligence', 'tweet': '"Exploring how #ArtificialIntelligence is reshaping our daily grind. It\'s revolutionizing work, learning, and social engagement. Can\'t wait to see what frontiers AI will conquer next! 🚀💻 #AIRevolution"\n\nStatus: APPROVED', 'noofiterations': 2, 'maximumiterations': 3}
